# Component 1 — Credit readiness: bank rules vs ML, and how much data is needed

Two research questions, both answerable on synthetic data because they compare methods against the
same labels (not claims about real-world accuracy):

1. **Rules vs ML vs hybrid.** Banks screen small shops with fixed rules (debt-to-income, time in
   business, stock-outs). Compared with the ML score, how many credit-ready shops do the rules turn
   away (*financial inclusion*), and how many approved shops are not credit-ready (*risk*)?
2. **Data needed.** How many shops' records must a lender have before the model is reliable?

Why not "how many *months* of history are needed"? `months_active` is part of the formula that
generated the labels (`digital_ledger_review.ipynb`, section 1), so that analysis would only give back
the formula. Training-set size does not have this problem.

The test set (500 shops) is the same as in `digital_ledger_review.ipynb` and is not used to choose
anything. The original notebooks, dataset and `.pkl` files are not changed.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42


def add_domain_features(X_df):
    # Same feature engineering as digital legder.ipynb / ml_service/app.py
    X_out = X_df.copy()
    rev = np.maximum(X_out['monthly_revenue_rs'], 1.0)
    exp = X_out['monthly_expenses_rs']
    X_out['net_cash_flow'] = rev - exp
    X_out['debt_to_income_ratio'] = exp / rev
    X_out['digital_revenue_volume'] = rev * X_out['digital_payment_ratio']
    X_out['revenue_per_active_month'] = rev / np.maximum(X_out['months_active'], 1.0)
    X_out['cash_flow_margin'] = X_out['net_cash_flow'] / rev
    return X_out

## 1. Same data, split and champion model as `digital_ledger_review.ipynb`

In [ ]:
df = pd.read_csv('sales & financial.csv').drop_duplicates().reset_index(drop=True)
df = df.drop(columns=['shop_id', 'year_month'], errors='ignore')
X_raw = df.drop(columns=['target_credit_ready'])
y = df['target_credit_ready'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
num_cols = add_domain_features(X_raw).select_dtypes('number').columns.tolist()

def make_model():
    pre = ColumnTransformer([('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)])
    return Pipeline([('feat_eng', FunctionTransformer(add_domain_features, validate=False)), ('prep', pre),
                     ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))])

model = make_model().fit(X_train, y_train)
p_ml = model.predict_proba(X_test)[:, 1]

def true_score(d):   # the label formula, used only for the oracle ceiling
    return (0.05 * d['profit_margin_pct'] + 0.09 * d['months_active'] + 2.0 * d['digital_payment_ratio']
            - 3.5 * d['stockout_rate'] - 1.5 * d['credit_sales_ratio'] - 0.15 * d['sales_volatility'])
p_oracle = 1 / (1 + np.exp(-(true_score(X_test) - np.median(true_score(X_raw)))))

y_true = y_test.to_numpy()
ORACLE_AUC = roc_auc_score(y_true, p_oracle)
print(f'Test set: {len(y_true)} shops, {y_true.mean():.0%} credit-ready')
print(f'ML test ROC-AUC {roc_auc_score(y_true, p_ml):.3f} | oracle ceiling {ORACLE_AUC:.3f}')

## 2. Experiment 1 — who gets approved?

| | Policy | Approves a shop when |
|---|---|---|
| R1 | Bank rules (live hard blocks) | debt-to-income ≤ 85 % **and** at least 3 months in business |
| R2 | Strict bank rules (original design) | R1 **and** stock-out rate ≤ 25 % |
| M | ML only | credit score ≥ 50 |
| H | **Live system** | R1 **and** credit score ≥ 50 (`ml_service/app.py`) |
| ★ | Oracle | the true formula says ≥ 50 % (ceiling) |

What each number means for a lender and for shop owners:

- **Approval rate**: share of shops that get a loan offer.
- **Precision** (*good-loan rate*): of the approved shops, the share that are really credit-ready.
- **Recall** (*inclusion*): of the credit-ready shops, the share that get an offer.
- **F1** balances the two. **Ready but rejected** = credit-ready shops turned away.

In [ ]:
eng = add_domain_features(X_test)
rule_basic = ((eng['debt_to_income_ratio'] <= 0.85) & (X_test['months_active'] >= 3)).to_numpy()
rule_strict = rule_basic & (X_test['stockout_rate'] <= 0.25).to_numpy()

policies = {
    'R1. Bank rules':          rule_basic,
    'R2. Strict bank rules':   rule_strict,
    'M. ML only':              p_ml >= 0.5,
    'H. Live: rules AND ML':   rule_basic & (p_ml >= 0.5),
    '★ Oracle':                (p_oracle >= 0.5).to_numpy(),
}

def scores(y, approve):
    tn, fp, fn, tp = confusion_matrix(y, approve, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {'approval rate %': approve.mean() * 100, 'precision': precision, 'recall': recall,
            'F1': 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            'ready but rejected': fn, 'approved but not ready': fp}

results = pd.DataFrame({k: scores(y_true, v) for k, v in policies.items()}).T
results = results.astype(float).round(3)
results

### Is the difference real? Bootstrap 95 % confidence intervals (1,000 resamples of the test shops)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.integers(0, len(y_true), size=(1000, len(y_true)))
yb = y_true.astype(bool)[idx]

def f1_of(f):
    fb = np.asarray(f, dtype=bool)[idx]
    tp = (yb & fb).sum(1); fp = (~yb & fb).sum(1); fn = (yb & ~fb).sum(1)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

boot = {k: f1_of(v) for k, v in policies.items()}
def show(a, b):
    d = boot[a] - boot[b]; lo, hi = np.percentile(d, [2.5, 97.5])
    print(f'F1({a}) - F1({b}): {d.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')

show('M. ML only', 'R1. Bank rules')
show('M. ML only', 'R2. Strict bank rules')
show('H. Live: rules AND ML', 'M. ML only')

## 3. Experiment 2 — how many shops' records are needed to train the model?

The model is retrained on random subsets of the training shops (each size 30 times) and always
tested on the same 500 test shops. Result shown as a share of the oracle ceiling.

In [ ]:
sizes = [25, 50, 100, 200, 400, 800, 1200, 2000]
rows = []
for n in sizes:
    aucs = []
    for rep in range(30 if n < len(X_train) else 1):
        if n < len(X_train):
            Xs, _, ys, _ = train_test_split(X_train, y_train, train_size=n, stratify=y_train, random_state=rep)
        else:
            Xs, ys = X_train, y_train
        aucs.append(roc_auc_score(y_true, make_model().fit(Xs, ys).predict_proba(X_test)[:, 1]))
    aucs = np.array(aucs)
    rows.append({'training shops': n, 'mean AUC': aucs.mean(), 'worst AUC (5th pct)': np.percentile(aucs, 5),
                 'mean % of oracle': aucs.mean() / ORACLE_AUC * 100})
curve = pd.DataFrame(rows).set_index('training shops').round(3)
curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

order = ['R1. Bank rules', 'R2. Strict bank rules', 'M. ML only', 'H. Live: rules AND ML']
r = results.loc[order]
x = np.arange(len(order)); w = 0.38
ax[0].bar(x - w / 2, r['precision'], w, label='Precision (good-loan rate)', color='#1f5fa8')
ax[0].bar(x + w / 2, r['recall'], w, label='Recall (inclusion)', color='#e08e79')
ax[0].set_xticks(x); ax[0].set_xticklabels([o.split('. ')[1] for o in order], rotation=12)
ax[0].set_ylim(0, 1.15); ax[0].legend(loc='upper center', ncol=2); ax[0].set_title('Who gets approved (test set, 500 shops)')

ax[1].plot(curve.index, curve['mean AUC'], 'o-', color='#1f5fa8', label='Mean test AUC')
ax[1].plot(curve.index, curve['worst AUC (5th pct)'], 's--', color='#888', label='Worst 5% of runs')
ax[1].axhline(ORACLE_AUC, color='#2e7d32', lw=1, label=f'Oracle ceiling ({ORACLE_AUC:.3f})')
ax[1].set_xscale('log'); ax[1].set_xticks(sizes); ax[1].set_xticklabels(sizes)
ax[1].set_xlabel('Number of training shops (log scale)'); ax[1].set_ylabel('ROC-AUC')
ax[1].legend(loc='lower right'); ax[1].set_title('How many shops are needed to train')
plt.tight_layout(); plt.savefig('component1_rules_vs_ml.png', dpi=120); plt.show()

## 4. Conclusion

*Written after reading the numbers above — fill in with the actual results:*

- Bank rules alone: approve ___ %, turn away ___ credit-ready shops of ___. ML: ___.
- The live system (rules AND ML): precision ___, recall ___ — compared with ML only: ___ (interval ___).
- The strict stock-out rule would turn away ___ credit-ready shops; this is why it is a notice, not a block.
- About ___ training shops are enough to reach ___ % of the ceiling.

**Limitations.** The labels come from a known formula with noise, so the absolute scores are about this
synthetic data, not real loan repayment. The comparisons (rules vs ML, data size) use the same labels
for every method and are valid as comparisons. A real study would need repayment outcomes from a lender.